In [1]:
%pip install rasterio
%pip install torch torchvision
%pip install segmentation-models-pytorch
%pip install rasterio
%pip install matplotlib
%pip install optuna
%pip install albumentations
%pip install --upgrade typing_extensions

  Using cached rasterio-1.4.3-cp314-cp314-macosx_15_0_arm64.whl
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached attrs-25.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached certifi-2025.1.31-py3-none-any.whl.metadata (2.5 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
  Using cached numpy-2.2.4-cp314-cp314-macosx_15_0_arm64.whl
  Using cached click_plugins-1.1.1-py2.py3-none-any.whl.metadata (6.4 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
Using cached affine-2.4.0-py3-none-any.whl (15 kB)
Using cached attrs-25.3.0-py3-none-any.whl (63 kB)
Using cached certifi-2025.1.31-py3-none-any.whl (166 kB)
Using cached click_plugins-1.1.1-py2.py3-none-any.whl (7.5 kB)
Using cached pyparsing-3.2.3-py3-none-any.whl (111 kB)
Note: you may need to restart the kernel

In [1]:
# Import necessary modules
import torch
from torch.utils.data import DataLoader
import optuna

# Import our custom modules.
import utils.data_io
import utils.augmentation
import utils.dataset_module
import utils.training_module
import utils.data_preparation
from utils.optuna_objective import objective

# Check device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

/Users/mimja/Development/hackathon/csu-2025/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Loading: ./South_Clear_Creek/Lidar_DEM_Hillshade/South_Clear_Creek_BareEarth_Hillshade_1m.tif
  Count: 1, dtype: uint8
  Bounds: BoundingBox(left=431955.0, bottom=4382622.0, right=442415.0, top=4395737.0)
  Shape: (1, 13115, 10460)
  NoData: (None,)
Loading: ./South_Clear_Creek/Roads_Boundary/South_Clear_Creek_Roads_Mask.tif
  Count: 1, dtype: float32
  Bounds: BoundingBox(left=431955.0, bottom=4382622.0, right=442415.0, top=4395737.0)
  Shape: (1, 13115, 10460)
  NoData: (nan,)
Loading: ./South_Clear_Creek/Lidar_DEM_Hillshade/South_Clear_Creek_BareEarth_DEM_1m.tif
  Count: 1, dtype: float32
  Bounds: BoundingBox(left=431955.0, bottom=4382622.0, right=442415.0, top=4395737.0)
  Shape: (1, 13115, 10460)
  NoData: (3.3999999521443642e+38,)

--- RAW DATA STATS ---
Hillshade -> min: 0, max: 238, mean: 100.01554005637729
Roads -> min: 0.0, max: 1.0, mean: 0.0007232752977870405


/Users/mimja/Development/hackathon/csu-2025/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


DEM -> min: 2587.711669921875, max: 3.3999999521443642e+38, mean: inf

--- AFTER CLAMPING ---
Hillshade: min 0.0, max 238.0, mean 100.01545715332031
Roads: min 0.0, max 1.0, mean 0.0007232752977870405
DEM: min 2587.711669921875, max 4210.41943359375, mean 3526.87744140625

Image dimensions: H=13115, W=10460
Number of patches extracted: 2040
Dataset size before oversampling: 2040 patches
Before oversampling: 29 positive, 2011 negative patches.
Each positive patch will be duplicated 68 times, with 10 additional copies needed.
After oversampling: 2011 positive, 2011 negative patches.
Total patches after oversampling: 4022
Dataset class distribution: 2011 positive, 2011 negative patches.
Dataset class distribution: 2011 positive, 2011 negative patches.
Data augmentation: total dataset size after original+augmentation: 8044
Total dataset size after augmentation: 8044 samples
Dataset split -> Train: 5630, Val: 1206, Test: 1208
Using device: cpu


In [ ]:
# Optionally, run an Optuna study to find the best hyperparameters.

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=1)

print("Best trial:")
trial = study.best_trial
print("  Loss:", trial.value)
print("  Hyperparameters:", trial.params)


[I 2025-03-29 14:07:08,199] A new study created in memory with name: no-name-630d156e-ee68-460d-a657-ee4214f806b1
Downloading: "http://data.lip6.fr/cadene/pretrainedmodels/densenet121-fbdb23505.pth" to /Users/mimja/.cache/torch/hub/checkpoints/densenet121-fbdb23505.pth
100%|██████████| 30.9M/30.9M [00:02<00:00, 11.6MB/s]



========== START TRAINING ==========

=== EPOCH 1/1 ===

Batch 1: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 1.1606

Batch 2: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 1.0277

Batch 3: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 0.9217

Batch 4: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 0.8255

Batch 5: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 0.7470

Batch 6: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 0.6963

Batch 7: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 0.6553

Batch 8: inputs.shape: torch.Size([67, 3, 256, 256]), targets.shape: torch.Size([67, 256, 256])
  Loss: 0.6240

Batch 9: inputs.shape: torch.Size([67, 3, 256,

In [ ]:
# Use predetermined hyperparameters (or use trial.params from Optuna)
import utils.data_preparation
hyperparams = {
    "lr": .0005167333,
    "batch_size": 79,
    "epochs": 50  # For many epochs; early stopping will use validation performance.
}

# Prepare data.
train_ds, val_ds, test_ds = utils.data_preparation.prepare_data(oversample=True)
train_loader = DataLoader(train_ds, batch_size=hyperparams["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=hyperparams["batch_size"], shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=hyperparams["batch_size"], shuffle=False, num_workers=0)

# Get model, loss, and optimizer.
model, loss_fn, optimizer = utils.training_module.get_model(lr=hyperparams["lr"])

# Train with early stopping.
utils.training_module.train_with_early_stopping(model, train_loader, val_loader, loss_fn, optimizer,
                                            num_epochs=hyperparams["epochs"], patience=3, 
                                            model_save_path="best_model.pth")

# Evaluate on validation and test sets.
print("\n--- Evaluation on Validation Set ---")
utils.training_module.evaluation_loop(model, val_loader, loss_fn)

print("\n--- Evaluation on Test Set ---")
utils.training_module.evaluation_loop(model, test_loader, loss_fn)

# Save predictions (e.g., for half of the test dataset).
num_samples = len(test_ds) // 2
indices_to_save = list(range(num_samples))
utils.training_module.save_predictions(model, test_ds, save_dir="predictions", indices=indices_to_save)
